# Session 5 — Random Sampling & File I/O

Runnable code for the **Build It** and **Experiment** sections. Run each cell top to bottom.

## 1. Build It — Core Code

In [1]:
import numpy as np

# ---------- 1. Random sampling from the Normal distribution ----------
heights = np.random.normal(170, 5, size=(3, 4))   # mean=170, std=5, 12 samples
print("Simulated heights:\n", heights)
print("Sample mean:", heights.mean(), "| Sample std:", heights.std())

Simulated heights:
 [[175.85049133 172.62099168 174.46903073 168.00355095]
 [169.01289961 172.2209699  170.68005721 171.59388873]
 [168.46048137 169.60808394 167.5698414  161.38020491]]
Sample mean: 170.12254097879114 | Sample std: 3.6115091812044553


In [2]:
# ---------- 2. Random sampling from the Uniform distribution ----------
grades = np.random.uniform(10, 20, size=50)   # 50 samples, evenly likely between 10 and 20
print("\nSample grades (first 10):", grades[:10])


Sample grades (first 10): [15.67459798 11.9401802  10.54857456 15.82790381 15.43448898 16.72983677
 15.58474173 16.28159724 19.66514216 14.64848055]


In [3]:
# ---------- 3. Reproducibility with a seed ----------
np.random.seed(42)
run_1 = np.random.normal(0, 1, size=5)
np.random.seed(42)
run_2 = np.random.normal(0, 1, size=5)
print("\nRun 1:", run_1)
print("Run 2 (same seed):", run_2)
print("Identical?", np.array_equal(run_1, run_2))


Run 1: [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
Run 2 (same seed): [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
Identical? True


In [4]:
# ---------- 4. Saving and loading a binary .npz file ----------
x = np.arange(10)
y = x ** 2

np.savez("np_file.npz", x_arr=x, y_arr=y)
del x, y   # simulate a fresh session where x, y no longer exist

loaded = np.load("np_file.npz")
x = loaded["x_arr"]
y = loaded["y_arr"]
loaded.close()   # release the file handle so the file can be removed later
print("\nReloaded from .npz -> x:", x, "y:", y)


Reloaded from .npz -> x: [0 1 2 3 4 5 6 7 8 9] y: [ 0  1  4  9 16 25 36 49 64 81]


In [5]:
# ---------- 5. Saving and loading a human-readable .csv file ----------
array_2d = np.hstack((x[:, np.newaxis], y[:, np.newaxis]))
np.savetxt("xy.csv", array_2d, header="x,y", delimiter=",")

del x, y
loaded_xy = np.loadtxt("xy.csv", delimiter=",")
x = loaded_xy[:, 0]
y = loaded_xy[:, 1]
print("\nReloaded from .csv -> x:", x, "y:", y)
print("Note: dtype is now float, not int ->", x.dtype)


Reloaded from .csv -> x: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.] y: [ 0.  1.  4.  9. 16. 25. 36. 49. 64. 81.]
Note: dtype is now float, not int -> float64


## 2. Experiment

In [6]:
# Experiment 1: sample size vs. how close statistics get to the true distribution
np.random.seed(0)
for n in [10, 1_000, 1_000_000]:
    sample = np.random.normal(170, 5, size=n)
    print(f"n={n:>9} -> mean={sample.mean():.3f}, std={sample.std():.3f}")
# Expect: larger n -> mean closer to 170, std closer to 5

n=       10 -> mean=173.690, std=4.835
n=     1000 -> mean=169.739, std=4.909
n=  1000000 -> mean=170.008, std=5.000


In [7]:
# Experiment 2: Normal vs Uniform -- visibly different spread patterns
normal_sample = np.random.normal(0, 1, size=10)
uniform_sample = np.random.uniform(-1, 1, size=10)
print("\nNormal sample (clusters near 0):", normal_sample)
print("Uniform sample (evenly spread -1 to 1):", uniform_sample)


Normal sample (clusters near 0): [ 0.53693824 -1.14904779 -0.44139613 -0.45157058  0.76168291 -0.81958453
  0.82900701  0.33219807  1.02329055 -1.83198958]
Uniform sample (evenly spread -1 to 1): [ 0.58539029 -0.15995889  0.17906266  0.50236147  0.39784191  0.75373832
  0.0173411   0.49817884  0.77912892 -0.81840447]


In [8]:
# Experiment 3: same seed -> same "random" numbers, different seed -> different numbers
np.random.seed(1)
a = np.random.uniform(0, 1, size=3)
np.random.seed(2)
b = np.random.uniform(0, 1, size=3)
np.random.seed(1)
c = np.random.uniform(0, 1, size=3)
print("\nseed=1:", a)
print("seed=2:", b)
print("seed=1 again:", c)
print("a equals c (same seed)?", np.array_equal(a, c))


seed=1: [4.17022005e-01 7.20324493e-01 1.14374817e-04]
seed=2: [0.4359949  0.02592623 0.54966248]
seed=1 again: [4.17022005e-01 7.20324493e-01 1.14374817e-04]
a equals c (same seed)? True


In [9]:
# Experiment 4: integer dtype lost after round-tripping through CSV
ints = np.array([1, 2, 3, 4], dtype=int)
np.savetxt("ints.csv", ints, delimiter=",")
reloaded_ints = np.loadtxt("ints.csv", delimiter=",")
print("\nOriginal dtype:", ints.dtype)
print("Reloaded dtype:", reloaded_ints.dtype)
print("Reloaded values:", reloaded_ints)  # now floats, e.g. [1. 2. 3. 4.]
print("Cast back to int:", reloaded_ints.astype(int))


Original dtype: int64
Reloaded dtype: float64
Reloaded values: [1. 2. 3. 4.]
Cast back to int: [1 2 3 4]


## 3. Mini Project — Simulate, Persist, and Round-Trip a Dataset

In [10]:
import os

# Step 1: simulate sensor readings
np.random.seed(7)
true_mean, true_std = 25.0, 2.5
sensor_readings = np.random.normal(true_mean, true_std, size=1000)

# Step 2: compare sample statistics to true parameters
print("True mean/std:", true_mean, true_std)
print("Sample mean/std:", sensor_readings.mean(), sensor_readings.std())

# Step 3: save full dataset in binary format
np.savez("sensor_data.npz", sensor_readings=sensor_readings)

# Step 4: reload and verify integrity
reloaded = np.load("sensor_data.npz")
reloaded_readings = reloaded["sensor_readings"]
reloaded.close()   # release the file handle so the file can be removed later
print("\nRound-trip via .npz identical?", np.array_equal(sensor_readings, reloaded_readings))

# Step 5: save a small human-readable subset as CSV
subset = sensor_readings[:20]
np.savetxt("sensor_subset.csv", subset, header="reading", delimiter=",")

reloaded_subset = np.loadtxt("sensor_subset.csv", delimiter=",")
print("\nReloaded CSV subset:", reloaded_subset)
print("Reloaded dtype:", reloaded_subset.dtype)

True mean/std: 25.0 2.5
Sample mean/std: 24.92881243357207 2.406149174232953

Round-trip via .npz identical? True

Reloaded CSV subset: [29.22631426 23.83515657 25.08205041 26.01879071 23.02769243 25.00516393
 24.99777404 20.61318923 27.54414501 26.50124629 23.43642757 24.57112935
 26.26324844 24.34660896 24.3931273  21.36689647 26.38645078 25.30970226
 25.68614981 21.18368867]
Reloaded dtype: float64


In [11]:
# Clean up generated files so the repository stays clean
for f in ("np_file.npz", "xy.csv", "ints.csv", "sensor_data.npz", "sensor_subset.csv"):
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up generated files.")

Cleaned up generated files.
